<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.6-vllm-and-gke-production/notebooks/exercises-11.6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.6 — vLLM & GKE Production
Netsetos GenAI Course v2.0 | Module 11: Cost & Performance

PagedAttention, continuous batching, multi-GPU, GKE node pools, observability, India hybrid (GKE + E2E).


In [ ]:
print('Module 11: Cost & Performance')
print('Lesson 11.6: vLLM & GKE Production')


## Ex 1: vLLM Throughput vs Naive HF


In [ ]:
print('Llama-3.1-8B throughput on H100 (illustrative):')
for concurrency, naive_hf, vllm in [
    (1,   52,  180),
    (4,   55,  410),
    (8,   60,  640),
    (16,  60,  780),
    (32,  60,  824),
    (64,  60,  830),  # plateau
]:
    gain = vllm / naive_hf
    print(f'  conc={concurrency:>3} | naive {naive_hf:>3} tok/s | vllm {vllm:>4} tok/s | gain {gain:>5.1f}x')
print()
print('Continuous batching breaks the sequential wall. Plateau at GPU saturation.')


## Ex 2: Inference Server Landscape


In [ ]:
print('Inference servers compared:')
for server, strength, best_for in [
    ('vLLM',          'PagedAttention, continuous batching, Python', 'default OSS, fast to deploy'),
    ('TGI (HF)',      'Easy multi-GPU, HF ecosystem',                'HF model deployments'),
    ('TensorRT-LLM',  'Max NVIDIA throughput, compiled',             'latency-critical saturated H100'),
    ('Triton',        'Multi-framework, ensemble graphs',            'mixed model types in one server'),
    ('Ray Serve',     'Python-native actor model',                   'multi-model + Python glue code'),
]:
    print(f'  {server:14s}: {strength:46s} | best: {best_for}')


## Ex 3: Multi-GPU Strategies


In [ ]:
print('Multi-GPU in vLLM:')
for strat, flag, when, link in [
    ('Single GPU',        '(default)',                'fits in 1 GPU',          'any'),
    ('Tensor Parallel',   '--tensor-parallel-size 4', 'doesn\'t fit, in 1 node','NVLink required'),
    ('Pipeline Parallel', '--pipeline-parallel-size 2','across nodes',           'Ethernet/InfiniBand OK'),
    ('TP + PP',            'TP 8 + PP 2',              '405B model, multi-node', 'NVLink + fast network'),
    ('Expert Parallel',    '--moe-expert-parallel',    'MoE models (Mixtral)',   'per-expert sharding'),
]:
    print(f'  {strat:18s}: {flag:26s} | when: {when:24s} | link: {link}')


## Ex 4: GKE GPU Configuration


In [ ]:
print('GKE GPU node pool sizing:')
for workload, gpu, count, notes in [
    ('Llama-3.1-8B   serving','L4 (24 GB)',     1, 'fits vLLM Q8'),
    ('Llama-3.1-8B   heavy',  'A100 (40 GB)',   1, 'headroom for long context'),
    ('Llama-3.1-70B',         'A100 (80 GB)',   4, 'TP=4 within node, NVLink'),
    ('Mixtral-8x22B',          'A100 (80 GB)',  4, 'MoE expert parallel'),
    ('Llama-3.1-405B',         'H100 (80 GB)',  8, 'TP=8 in node + PP=2 across'),
]:
    print(f'  {workload:22s}: {gpu:16s} x{count} | {notes}')


## Ex 5: vLLM Prometheus Metrics


In [ ]:
print('Essential vLLM metrics:')
for metric, meaning, alert_threshold in [
    ('vllm:num_requests_running',        'concurrent decodes',       '> 0.95 * max_num_seqs -> scale'),
    ('vllm:num_requests_waiting',        'queue depth',              '> 50 -> autoscale fire'),
    ('vllm:gpu_cache_usage_perc',        'KV cache fill',            '> 0.90 -> memory pressure'),
    ('vllm:request_latency',             'end-to-end',               'p99 > SLO -> investigate'),
    ('vllm:time_to_first_token',         'TTFT',                     'p99 > 500 ms -> issue'),
    ('vllm:time_per_output_token',       'TPOT',                     'regression from baseline'),
    ('vllm:num_preemptions_total',       'OOM preempts',             '> 0 -> cache too small'),
]:
    print(f'  {metric:36s} | {meaning:24s} | alert: {alert_threshold}')


## Ex 6: Cold Start Mitigations


In [ ]:
print('Cold start time for model load:')
for model, size_gb, bake_image_s, pvc_s, streaming_s in [
    ('Llama-3.1-8B',   16, 35,  8, 12),
    ('Llama-3.1-70B', 140,320, 55, 70),
    ('Mixtral-8x22B', 280,620,110,130),
]:
    print(f'  {model:18s} ({size_gb:>3} GB) | baked img: {bake_image_s}s | PVC: {pvc_s}s | stream: {streaming_s}s')
print()
print('Production: min-replicas >= 1 so user-facing never pays cold start.')


## Ex 7: India GPU Cost Stack


In [ ]:
print('India GPU pricing per hour (spot / on-demand, 2026):')
for gpu, e2e_inr, yotta_inr, gke_asia_south, indiaai in [
    ('L4 24GB',   None,    None,    120,  None),
    ('A100 40GB', 150,     180,     280,   65),
    ('A100 80GB', 220,     260,     340,   85),
    ('H100 80GB', 88,      105,     500,  150),  # E2E spot headline
    ('H200',      None,    140,     None, None),
]:
    print(f'  {gpu:12s} | E2E {str(e2e_inr):>4} | Yotta {str(yotta_inr):>4} | GKE {str(gke_asia_south):>4} | IndiaAI {str(indiaai):>4} (INR/hr)')
print()
print('E2E + IndiaAI cheapest. GKE wins on managed + global APIs.')


---
## Recap
vLLM PagedAttention + continuous batching = 16x throughput vs naive HF.
Multi-GPU: TP within node (NVLink), PP across nodes. v2 block manager for prefix sharing.
GKE Standard + GPU node pool + spot 70%. Autoscale on vllm:num_requests_waiting, not CPU.
Cold start: min-replicas >= 1 for user-facing. Bake image or PVC for model weights.
India: hybrid GKE asia-south1 burst + E2E H100 Rs88/hr steady baseline.
